- $p_\star(x)$ 是概率密度，$f(x)$ 任意函数，蒙特卡洛近似求 $\mathbb E_{p_\star(x)}[f(x)]$
    - 基于概率分布 $p_\star(x)$ 采样样本：$\left\{x^{(1)},x^{(2)},x^{(3)},\cdots,x^{(N)}\right\}$
    - 求出每个样本 $x^{(i)}$ 对应的 $f(x^{(i)})$，计算均值

## kld => mle

$$
 \arg\min_\theta D_{KL}(p_\star || p_\theta) \iff \arg\max_\theta \mathbb{E}_{x \sim p_\star}[\log p_\theta(x)] \approx \arg\max\sum_{n=1}^N\log p_\theta(x_n)
$$

- forward kl: mean seeking, zero-avoiding
    - $p_\star\gt 0 \Rightarrow p_\theta\gt 0$
    - 如果是 reverse KL 的话，没有下文的性质

$$
\begin{aligned}
D_{KL}(p_\star || p_\theta) &= \int p_\star(x) \log \frac{p_\star(x)}{p_\theta(x)} dx \\
&= \int p_\star(x) (\log p_\star(x) - \log p_\theta(x)) dx \\
&= \int p_\star(x) \log p_\star(x) dx - \int p_\star(x) \log p_\theta(x) dx \\
&= \mathbb{E}_{x \sim p_\star}[\log p_\star(x)] - \mathbb{E}_{x \sim p_\star}[\log p_\theta(x)] \\
&= -H(p_\star) - \mathbb{E}_{x \sim p_\star}[\log p_\theta(x)]
\end{aligned}
$$

- MLE 的蒙特卡洛近似
    - $ \mathbb{E}_{x \sim p_\star}[\log p_\theta(x)] \approx \frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i) $

## 引入隐变量

- $\log p_\theta(x)=\log\int p_\theta(x,z)dz=\log\sum_z p_\theta(x,z)$
    - 考虑样本集 $\mathcal D=\left\{x^{(1)},x^{(2)},x^{(3)},\cdots,x^{(N)}\right\}$
    - 则整个样本集的对数似然为

$$
\begin{split}
\log p_\theta(\mathcal D)&=\log\left(\Pi_{i=1}^Np_\theta(x^{(i)})\right)\\
&=\sum_{i=1}^N\log p_\theta(x^{(i)})\\
&=\sum_{i=1}^N\log\sum_{z^{(i)}}p_\theta(x^{(i)},z^{(i)})
\end{split}
$$

- 现在的式子是 log-sum 的形式，没有解析解；
- 考虑单个数据的对数似然 $\log p_\theta(x)$
    - 几种等价形式
        - $\log p_\theta(x)=\log\sum_{z}p_\theta(x,z)$
        - $\log p_\theta(x)=\log\frac{p_\theta(x,z)}{p_\theta(z|x)}$
            - 但对于 $p_\theta(z|x)$（posterior），其 $p_\theta(z|x)=\frac{p_\theta(x,z)}{\sum_zp_\theta(x,z)}$

### 引入 $q(z)$

- $q(z)$ 来近似 $p_\theta(z|x)$

$$
\begin{aligned}
\log p_{\theta}(x) &= \log p_{\theta}(x) \sum_{z} q(z) \\
&= \sum_{z} \left(q(z) \log p_{\theta}(x)\right) \\
&=\sum_z q(z)\log\frac{p_\theta(x,z)}{p_\theta(z|x)}\\
&=\sum_z q(z)\log\frac{p_\theta(x,z)q(z)}{p_\theta(z|x)q(z)}\\
&= \sum_{z} q(z) \left( \log \frac{p_{\theta}(x,z)}{q(z)} + \log \frac{q(z)}{p_{\theta}(z|x)} \right) \\
&= \sum_{z} q(z) \log \frac{p_{\theta}(x,z)}{q(z)} + \sum_{z} q(z) \log \frac{q(z)}{p_{\theta}(z|x)} \\
&= \sum_{z} q(z) \log \frac{p_{\theta}(x,z)}{q(z)} + D_{KL}(q(z) || p_{\theta}(z|x))
\end{aligned}
$$

## EM

#### ELBO

$$
\log p_{\theta}(x)=\sum_{z} q(z) \log \frac{p_{\theta}(x,z)}{q(z)} + \underbrace{D_{KL}(q(z) || p_{\theta}(z|x))}_{\geq 0}
$$
- $\log p_{\theta}(x)\geq\sum_{z} q(z) \log \frac{p_{\theta}(x,z)}{q(z)}$
    - https://en.wikipedia.org/wiki/Evidence_lower_bound
- 定义 $\text{ELBO}(x;q,\theta)=\sum_zq(z)\log\frac{p_\theta(x,z)}{q(z)}$
    - 对数似然 $\log p_\theta(x)$ 总是大于或等于 $\text{ELBO}(x;q,\theta)$
    - 这里的 elbo 的形式是 sum-log 的形式，易于求解
    - kl + elbo 是固定的（关于 $q(z)$，无论 $q(z)$ 如何变化，elbo + kl 的总和不变）
        - kl 越小，elbo 就越大，当 $q(z)=p_\theta(z|x)$ 时，kl 为 0，elbo 最大

### EM

- $\text{ELBO}(x;q,\theta)$ 有两个参数 $q(z),\theta$
    - 同时优化 $q(z)$ 和 $\theta$ 并不容易，EM 的思路是重复/迭代进行固定一个参数并优化另一个参数的过程
    - 固定 $\theta$，优化 $q(z)$
    - 固定 $q(z)$，优化 $\theta$
- 考虑到无论 $q(z)$ 如何变化，elbo + kl 的总和不变
    - kl 越小，elbo 就越大，当 $q(z)=p_\theta(z|x)$ 时，kl 为 0，elbo 最大（等于对数似然，$\log p_\theta(x)$）
- 固定 $\theta=\theta_{old}$ 更新 $q(z)$ => E step (Expectation Value)
    - 我们可以将 $q(z)=p_{\theta_{old}}(z|x)$，
    - 此时的 ELBO 为

$$
\text{ELBO}(x;q=p_{\theta_{old}}(z|x))=\sum_z p_{\theta_{old}}(z|x)\log \frac{p_{\theta}(x,z)}{p_{\theta_{old}}(z|x)}
$$

- sum-log 的形式，可以用解析的方式求解；